# DoSA-Maxwell 통합 자동화 튜토리얼

DoSA 설계 파일을 파싱한 후 **`apply_dosa_to_maxwell()`** 함수 한 번으로
Ansys Maxwell에 형상·재질·여자·설정을 모두 적용합니다.

```python
parse_dosa_file()        # DoSA .dsa → DesignModel
Maxwell2d / Maxwell3d()  # pyaedt 세션 생성 (pyaedt 예제와 동일)
apply_dosa_to_maxwell()  # DoSA → Maxwell 변환
```

> **AEDT 버전:** 2026.1 · **모드:** non-graphical · **단위:** mm


In [ ]:
from pathlib import Path
import sys
import importlib

# -- 경로 설정 --------------------------------------------------------------
REPO_ROOT  = Path(r"E:/KDH/gitDosa_Actuator/MoaActuatorBasedOnDoSA")
SAMPLE_DIR = REPO_ROOT / "DoSA-2D/Code/11_DoSA-2D/DoSA-2D/Samples"
OUTPUT_DIR = REPO_ROOT / "output"
OUTPUT_DIR.mkdir(exist_ok=True)

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# -- moa_actuator / dosa_maxwell 모듈 리로드 (소스 수정 후 재실행 시 반영) ----
for _mod in list(sys.modules):
    if _mod.startswith("moa_actuator") or _mod.startswith("dosa_maxwell"):
        importlib.reload(sys.modules[_mod])

# -- pyaedt ----------------------------------------------------------------
import ansys.aedt.core

# -- dosa_maxwell는 moa_actuator의 서브패키지 ---------------------------------
from moa_actuator.dosa_maxwell import parse_dosa_file, apply_dosa_to_maxwell, get_profile
from moa_actuator.dosa_maxwell.profiles import list_profiles

AEDT_VERSION = "2026.1"   # Maxwell 2026 R1
NG_MODE      = False       # headless (비그래픽)

print(f"ansys.aedt.core : {ansys.aedt.core.__version__}")
print(f"AEDT target     : {AEDT_VERSION}")
print(f"Output dir      : {OUTPUT_DIR}")

ansys.aedt.core : 1.0.0rc2
AEDT target     : 2026.1
Output dir      : E:\KDH\gitDosa_Actuator\DoSA-2D\Code\31_DoSA-Maxwell-Automation\output


## 1. DoSA 설계 파싱


In [42]:
design_sol = parse_dosa_file(SAMPLE_DIR / "Solenoid/Solenoid.dsa")
design_vcm = parse_dosa_file(SAMPLE_DIR / "VCM/VCM.dsa")

print(f"Solenoid parts : {[p.name for p in design_sol.parts]}")
print(f"         tests : {[t.name for t in design_sol.tests]}")
print()
print(f"VCM      parts : {[p.name for p in design_vcm.parts]}")
print(f"         tests : {[t.name for t in design_vcm.tests]}")


Solenoid parts : ['coil', 'plunger', 'core', 'case']
         tests : ['force', 'stroke', 'current']

VCM      parts : ['coil', 'magnet', 'plate', 'yoke']
         tests : ['force', 'stroke', 'current']


## 2. Solenoid — Maxwell 2D (MagnetostaticZ — RZ 축대칭)

pyaedt 예제와 동일하게 세션을 열고,
`apply_dosa_to_maxwell()` 한 번으로 DoSA 모델을 적용합니다.

> **주의**: pyAEDT에서 "about Z" 축대칭 모드는 `"MagnetostaticZ"` 문자열을 사용합니다.
> `"magnetostaticaboutz"`, `"MagnetostaticAboutZ"` 등은 유효하지 않습니다.

In [43]:
profile = get_profile("le01_2020r1")   # Magnetostatic 프리셋

# pyAEDT 유효 solution_type 문자열:
#   "MagnetostaticZ"  → about Z (RZ 축대칭) ← DoSA 솔레노이드용
#   "MagnetostaticXY" → XY 평면
#   "TransientZ"      → about Z 과도해석
#   "TransientXY"     → XY 과도해석
m2d_sol = ansys.aedt.core.Maxwell2d(
    design="Solenoid_2De",
    solution_type="MagnetostaticZ",
    version=AEDT_VERSION,
    non_graphical=NG_MODE,
    new_desktop=False,
)

PyAEDT INFO: Python version 3.10.11 (tags/v3.10.11:7d4cc5a, Apr  5 2023, 00:38:17) [MSC v.1929 64 bit (AMD64)].
PyAEDT INFO: PyAEDT version 1.0.0rc2.
PyAEDT INFO: Returning found Desktop session with PID 25944!
PyAEDT INFO: No project is defined. Project Solenoid exists and has been read.
PyAEDT INFO: Added design 'Solenoid_2De' of type Maxwell 2D.
PyAEDT INFO: AEDT objects correctly read


In [39]:
m2d_sol.solution_type

'Magnetostatic'

In [17]:
design_sol.tests[0].name

'force'

In [44]:
m2d_sol.modeler.model_units = "mm"

# DoSA -> Maxwell: 형상/재질/코일 여자/해석 설정/Force/Parametric Sweep/Report 모두 적용
result = apply_dosa_to_maxwell(design_sol, m2d_sol)

print(result)
print("\n--- Commands ---")
for c in result.get("commands", []):
    print(f"  {c}")
for e in result.get("errors", []):
    print(f" ! {e}")

PyAEDT INFO: Modeler2D class has been initialized!
PyAEDT INFO: Modeler class has been initialized! Elapsed time: 0m 0sec
PyAEDT INFO: Materials class has been initialized! Elapsed time: 0m 0sec
PyAEDT INFO: Boundary Vector Potential VectorPotential1 has been created.
PyAEDT INFO: Boundary Force Force_plunger has been created.
PyAEDT INFO: Boundary Current Current_coil has been created.
PyAEDT INFO: PostProcessor class has been initialized! Elapsed time: 0m 0sec
PyAEDT INFO: PostProcessor class has been initialized! Elapsed time: 0m 0sec
PyAEDT INFO: Post class has been initialized! Elapsed time: 0m 0sec
PyAEDT WARNING: No report category provided. Automatically identified Magnetostatic
<ApplyResult OK | parts=4 errors=0>

--- Commands ---
  plane_check: about-Z detected → geometry will be placed on ZX plane
  modeler.create_polyline(coil, 4 pts, ZX plane)
  modeler.create_polyline(plunger, 4 pts, ZX plane)
  modeler.create_polyline(core, 6 pts, ZX plane)
  modeler.create_polyline(case

In [ ]:
# 저장 후 Parametric Sweep 해석 실행
out = OUTPUT_DIR / "solenoid_2de" / "Solenoid.aedt"
out.parent.mkdir(parents=True, exist_ok=True)
m2d_sol.save_project(str(out))
print("Saved:", m2d_sol.project_file)


PyAEDT INFO: Project Solenoid Saved correctly
Saved: E:\KDH\gitDosa_Actuator\DoSA-2D\Code\31_DoSA-Maxwell-Automation\output\solenoid_2de\Solenoid.aedt


In [47]:

RUN_SOLVE = True   # True로 바꾸면 해석 실행

if RUN_SOLVE:
    # Parametric sweep 실행 (ParametricSetup1이 있으면)
    try:
        m2d_sol.analyze_setup("ParametricSetup1")
        print("Parametric sweep done.")
    except Exception:
        # Parametric이 없으면 기본 Setup만 해석
        m2d_sol.analyze_setup("DoSA_Setup")
        print("Setup solve done.")

PyAEDT INFO: Solving Optimetrics
PyAEDT INFO: Design setup ParametricSetup1 solved correctly in 0.0h 2.0m 48.0s
Parametric sweep done.


In [33]:
design_sol.parts

[NodeModel(kind='Coil', name='coil', properties={'NodeName': 'coil', 'KindKey': 'COIL', 'MovingParts': 'FIXED', 'Material': 'Copper', 'CurrentDirection': 'IN', 'Turns': '1040', 'Resistance': '15.20945', 'Layers': '20', 'TurnsOfOneLayer': '52', 'CoilWireGrade': 'Enameled_IEC_Grade_2', 'InnerDiameter': '9.6', 'OuterDiameter': '21.6', 'Height': '16', 'CopperDiameter': '0.27', 'WireDiameter': '0.31072', 'Temperature': '20', 'HorizontalCoefficient': '0.9', 'VerticalCoefficient': '0.98', 'ResistanceCoefficient': '1'}, children=[NodeModel(kind='Shape', name='', properties={'BasePointX': '0', 'BasePointY': '0', 'FaceType': 'RECTANGLE', 'PointX': '4.8', 'PointY': '14', 'LineKind': 'STRAIGHT', 'ArcDriction': 'FORWARD'}, children=[], raw_lines=['BasePointX=0', 'BasePointY=0', 'FaceType=RECTANGLE', 'PointX=4.8', 'PointY=-2', 'LineKind=STRAIGHT', 'ArcDriction=FORWARD', 'PointX=10.8', 'PointY=-2', 'LineKind=STRAIGHT', 'ArcDriction=FORWARD', 'PointX=10.8', 'PointY=14', 'LineKind=STRAIGHT', 'ArcDricti

In [34]:
import numpy as np
import matplotlib.pyplot as plt

# --- 해석 완료 후 결과 추출 (RUN_SOLVE=True일 때만 유효) ---
# MOVING 파트의 Force_z 결과를 Amp × move 전체 variation에서 추출

# 코일 파트에서 Amp 변수명 결정
coil_parts = [p for p in design_sol.parts if p.kind == "Coil"]
amp_var = f"Amp_{coil_parts[0].name}" if coil_parts else "Amp_1"

# MOVING 파트에서 Force expression 결정
moving_parts = [p for p in design_sol.parts if p.properties.get("MovingParts") == "MOVING"]
force_expr = f"Force_{moving_parts[0].name}.Force_z" if moving_parts else "Force.Force_z"


In [35]:
force_expr

'Force_plunger.Force_z'

In [ ]:

try:
    # 1) Amp 축 값 탐색
    amp_probe = m2d_sol.post.get_solution_data(
        expressions=[force_expr],
        variations={amp_var: "All", "move": "All"},
        primary_sweep_variable=amp_var,
    )

    if amp_probe is None:
        raise RuntimeError("Solution data를 가져오지 못했습니다. 먼저 해석을 완료하세요.")

    amp_values_obj = getattr(amp_probe, "primary_sweep_values", None)
    amp_values_raw = list(amp_values_obj) if amp_values_obj is not None else []
    amp_values = list(dict.fromkeys(str(v) for v in amp_values_raw))

    if not amp_values:
        amp_values = ["500A", "1000A"]

    # 2) Amp 값별 move sweep 데이터 수집
    rows = []
    for amp in amp_values:
        sd = m2d_sol.post.get_solution_data(
            expressions=[force_expr],
            variations={amp_var: amp, "move": "All"},
            primary_sweep_variable="move",
        )
        if sd is None:
            continue
        x_vals = np.array(sd.primary_sweep_values, dtype=float)
        y_vals = np.array(sd.data_real(force_expr), dtype=float)
        for x, y in zip(x_vals, y_vals):
            rows.append({amp_var: amp, "move": float(x), force_expr: float(y)})

    if not rows:
        raise RuntimeError("Amp/move variation 데이터를 읽지 못했습니다.")

    # 표 출력
    try:
        import pandas as pd
        df = pd.DataFrame(rows).sort_values([amp_var, "move"]).reset_index(drop=True)
        display(df)
    except ImportError:
        print(rows)

    # 플롯
    plt.figure(figsize=(8, 5))
    amp_keys = sorted({r[amp_var] for r in rows})
    for amp in amp_keys:
        subset = sorted((r for r in rows if r[amp_var] == amp), key=lambda r: r["move"])
        x = [r["move"] for r in subset]
        y = [r[force_expr] for r in subset]
        plt.plot(x, y, marker="o", linewidth=1.8, label=f"{amp_var}={amp}")

    plt.title(f"{force_expr} vs move (all {amp_var} variations)")
    plt.xlabel("move [mm]")
    plt.ylabel(f"{force_expr} [N]")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()

except RuntimeError as e:
    print(f"⚠ {e}")
    print("→ RUN_SOLVE=True로 설정하고 해석을 먼저 실행하세요.")

## 3. VCM — Maxwell 2D (MagnetostaticXY)


In [ ]:
m2d_vcm = ansys.aedt.core.Maxwell2d(
    design="VCM_2D",
    solution_type="MagnetostaticZ",
    version=AEDT_VERSION,
    non_graphical=NG_MODE,
    new_desktop=False,
)
m2d_vcm.modeler.model_units = "mm"

result_vcm = apply_dosa_to_maxwell(design_vcm, m2d_vcm, profile=profile)
print(result_vcm)
for e in result_vcm.get("errors", []):
    print(" !", e)

out_vcm = OUTPUT_DIR / "vcm_2d" / "VCM.aedt"
out_vcm.parent.mkdir(parents=True, exist_ok=True)
m2d_vcm.save_project(str(out_vcm))
print("Saved:", m2d_vcm.project_file)
# m2d_vcm.release_desktop(close_projects=False, close_desktop=False)

## 4. Solenoid — Maxwell 3D (축대칭 → 360° 회전체)

DoSA 2D 단면(X=반경, Y=축)을 Y축 기준 360° 회전시켜 3D 솔리드를 생성합니다.
`apply_dosa_to_maxwell()`이 Maxwell3d 인스턴스를 받으면 자동으로 3D 처리합니다.


In [ ]:
m3d_sol = ansys.aedt.core.Maxwell3d(
    design="Solenoid_3D",
    solution_type="Magnetostatic",
    version=AEDT_VERSION,
    non_graphical=NG_MODE,
    new_desktop=False,
)
m3d_sol.modeler.model_units = "mm"

# Maxwell3d 감지 -> polyline 후 sweep_around_axis(Y, 360deg) 자동 적용
result_3d = apply_dosa_to_maxwell(design_sol, m3d_sol, profile=profile)
print(result_3d)
for e in result_3d.get("errors", []):
    print(" !", e)

out_sol3d = OUTPUT_DIR / "solenoid_3d" / "Solenoid_3D.aedt"
out_sol3d.parent.mkdir(parents=True, exist_ok=True)
m3d_sol.save_project(str(out_sol3d))
print("Saved:", m3d_sol.project_file)
# m3d_sol.release_desktop(close_projects=False, close_desktop=False)


## 5. VCM — Maxwell 3D


In [ ]:
m3d_vcm = ansys.aedt.core.Maxwell3d(
    design="VCM_3D",
    solution_type="Magnetostatic",
    version=AEDT_VERSION,
    non_graphical=NG_MODE,
    new_desktop=False,
)
m3d_vcm.modeler.model_units = "mm"

result_vcm3d = apply_dosa_to_maxwell(design_vcm, m3d_vcm, profile=profile)
print(result_vcm3d)
for e in result_vcm3d.get("errors", []):
    print(" !", e)

out_vcm3d = OUTPUT_DIR / "vcm_3d" / "VCM_3D.aedt"
out_vcm3d.parent.mkdir(parents=True, exist_ok=True)
m3d_vcm.save_project(str(out_vcm3d))
print("Saved:", m3d_vcm.project_file)
# m3d_vcm.release_desktop(close_projects=False, close_desktop=False)


## 6. PDF 기반 프로파일 시뮬레이션 (2020R1 + 2014)

추가된 PDF들에 대응되는 프로파일을 `unified_plan.json`에서 자동 탐색해
동일한 DoSA 모델에 적용하고 프로젝트를 생성합니다.

- 2020R1: `le01_2020r1`, `ws01_2020r1`
- 2014: `coupling_1114_2014`, `tpc_1210_2014`

필요하면 `RUN_SOLVE = True`로 바꿔 실제 해석까지 실행할 수 있습니다.


In [ ]:
PDF_ROOT = ROOT.parents[1]
RUN_SOLVE = False  # True로 변경하면 setup 해석까지 수행

all_profiles = [p for p in list_profiles() if p.get("source_pdf") and p["source_pdf"] != "internal"]
selected_profiles = []
for p in all_profiles:
    source_pdf = p["source_pdf"]
    pdf_path = PDF_ROOT / source_pdf
    if pdf_path.exists() and ("2020" in source_pdf or "2014" in source_pdf):
        selected_profiles.append((p["name"], pdf_path))

print("Detected PDF profiles:")
for name, path in selected_profiles:
    print(f" - {name}: {path.name}")

for i, (profile_name, pdf_path) in enumerate(selected_profiles):
    prof = get_profile(profile_name)

    print(f"\n=== {profile_name} | solution={prof.solution_type} ===")
    print(f"PDF exists: {pdf_path.exists()} -> {pdf_path}")

    m2d_pdf = ansys.aedt.core.Maxwell2d(
        design=f"Solenoid_2D_{profile_name}",
        solution_type=("TransientXY" if prof.solution_type == "Transient" else "MagnetostaticXY"),
        version=AEDT_VERSION,
        non_graphical=NG_MODE,
        new_desktop=(i == 0),
    )
    m2d_pdf.modeler.model_units = "mm"

    result_pdf = apply_dosa_to_maxwell(design_sol, m2d_pdf, profile=prof)
    print(result_pdf)
    for e in result_pdf.get("errors", []):
        print(" !", e)

    out_pdf = OUTPUT_DIR / "pdf_profiles" / f"Solenoid_{profile_name}.aedt"
    out_pdf.parent.mkdir(parents=True, exist_ok=True)
    m2d_pdf.save_project(str(out_pdf))
    print("Saved:", m2d_pdf.project_file)

    if RUN_SOLVE:
        try:
            m2d_pdf.analyze_setup("DoSA_Setup")
            print("Solve done: DoSA_Setup")
        except Exception as exc:
            print(f"Solve skipped/failed: {exc}")


## 7. 세션 정리 및 결과 확인


In [ ]:
# 공유 Desktop 세션 정리
# 참고: Desktop()를 직접 호출하면 활성 세션이 없을 때 새 세션이 열릴 수 있어,
# 이미 생성한 앱 객체에서 release_desktop()를 호출합니다.
apps = [
    globals().get("m2d_pdf"),
    globals().get("m3d_vcm"),
    globals().get("m3d_sol"),
    globals().get("m2d_vcm"),
    globals().get("m2d_sol"),
]

released = False
for app in apps:
    if app is None:
        continue
    try:
        app.release_desktop(close_projects=True, close_desktop=True)
    except TypeError:
        app.release_desktop()
    released = True
    print("AEDT desktop released.")
    break

if not released:
    print("No active AEDT app instance found; cleanup skipped.")

print("\n=== 생성된 .aedt 파일 ===")
for f in sorted(OUTPUT_DIR.rglob("*.aedt")):
    print(f"  {f}")
